# One-Click Pipeline: cWGAN-GP Generation → Merge Dataset → Train VGG19

This notebook loads `generator_epoch_180.pth`, generates synthetic images into `SYNTHETIC (cWGAN)`, creates a combined dataset, splits REAL images into 70/20/10, adds synthetic images only to train, and trains VGG19 using the specified hyperparameters.

Validation and testing remain REAL-only for fair evaluation.

In [ ]:
!pip install -q scikit-learn tqdm matplotlib

import os, glob, shutil, random
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torchvision.utils import save_image
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, roc_auc_score, roc_curve, auc
from sklearn.preprocessing import label_binarize
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(42)

In [ ]:
# =========================
# CONFIG - CHANGE THESE TWO PATHS
# =========================
REAL_DIR = "/kaggle/input/your-dataset/Philippine Medicinal Plant Leaf Dataset/REAL"
GENERATOR_CHECKPOINT = "/kaggle/input/your-cwgan-output/cwgan_gp_outputs/saved_generators/generator_epoch_180.pth"

OUTPUT_DIR = "/kaggle/working/one_click_cwgan_vgg19"
SYNTHETIC_DIR = os.path.join(OUTPUT_DIR, "SYNTHETIC (cWGAN)")
COMBINED_DIR = os.path.join(OUTPUT_DIR, "COMBINED_REAL_cWGAN")
MODEL_DIR = os.path.join(OUTPUT_DIR, "models")
PLOT_DIR = os.path.join(OUTPUT_DIR, "plots")
for d in [OUTPUT_DIR, SYNTHETIC_DIR, COMBINED_DIR, MODEL_DIR, PLOT_DIR]: os.makedirs(d, exist_ok=True)

latent_dim = 100
gan_img_size = 64
channels = 3
synthetic_per_class = 60

train_ratio, val_ratio, test_ratio = 0.70, 0.20, 0.10

cnn_img_size = 224
batch_size = 16
epochs = 60
learning_rate = 0.0001
weight_decay = 0.0001
dropout_rate = 0.10
frozen_layers = 12
optimizer_name = "Adam"

print("REAL_DIR:", REAL_DIR)
print("GENERATOR_CHECKPOINT:", GENERATOR_CHECKPOINT)

In [ ]:
# =========================
# cWGAN-GP GENERATOR ARCHITECTURE
# Must match your upgraded cWGAN-GP generator.
# =========================
class Generator(nn.Module):
    def __init__(self, latent_dim, num_classes, channels):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, num_classes)
        input_dim = latent_dim + num_classes
        self.project = nn.Sequential(
            nn.Linear(input_dim, 512 * 4 * 4),
            nn.BatchNorm1d(512 * 4 * 4),
            nn.ReLU(True)
        )
        self.model = nn.Sequential(
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.ConvTranspose2d(64, channels, 4, 2, 1, bias=False), nn.Tanh()
        )
    def forward(self, noise, labels):
        x = torch.cat([noise, self.label_emb(labels)], dim=1)
        x = self.project(x).view(x.size(0), 512, 4, 4)
        return self.model(x)

def denorm(x): return (x + 1) / 2

In [ ]:
# =========================
# LOAD GENERATOR
# =========================
assert os.path.exists(REAL_DIR), f"REAL_DIR not found: {REAL_DIR}"
assert os.path.exists(GENERATOR_CHECKPOINT), f"Generator checkpoint not found: {GENERATOR_CHECKPOINT}"

real_dataset_for_classes = datasets.ImageFolder(root=REAL_DIR)
class_names = real_dataset_for_classes.classes
num_classes = len(class_names)
print("Detected classes:", num_classes)
print("First 5 classes:", class_names[:5])

G = Generator(latent_dim, num_classes, channels).to(device)
checkpoint = torch.load(GENERATOR_CHECKPOINT, map_location=device)
if isinstance(checkpoint, dict) and "G_state_dict" in checkpoint:
    G.load_state_dict(checkpoint["G_state_dict"])
elif isinstance(checkpoint, dict) and "generator_state_dict" in checkpoint:
    G.load_state_dict(checkpoint["generator_state_dict"])
else:
    G.load_state_dict(checkpoint)
G.eval()
print("Generator loaded successfully.")

In [ ]:
# =========================
# GENERATE SYNTHETIC IMAGES PER CLASS
# =========================
@torch.no_grad()
def generate_synthetic_images(images_per_class=60):
    print("Generating synthetic images...")
    for class_idx, class_name in enumerate(tqdm(class_names, desc="Classes")):
        class_folder = os.path.join(SYNTHETIC_DIR, class_name)
        os.makedirs(class_folder, exist_ok=True)
        for old_file in glob.glob(os.path.join(class_folder, "*.png")): os.remove(old_file)
        generated, gen_batch_size = 0, 32
        while generated < images_per_class:
            current_bs = min(gen_batch_size, images_per_class - generated)
            z = torch.randn(current_bs, latent_dim, device=device)
            labels = torch.full((current_bs,), class_idx, dtype=torch.long, device=device)
            fake_imgs = denorm(G(z, labels)).clamp(0, 1)
            for i in range(current_bs):
                save_path = os.path.join(class_folder, f"{class_name}_cwgan_{generated+i+1:03d}.png")
                save_image(fake_imgs[i], save_path)
            generated += current_bs
    print("Synthetic images saved to:", SYNTHETIC_DIR)

generate_synthetic_images(synthetic_per_class)

In [ ]:
# =========================
# CREATE COMBINED DATASET
# Train = REAL train + SYNTHETIC
# Val/Test = REAL only
# =========================
def reset_folder(path):
    if os.path.exists(path): shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)
reset_folder(COMBINED_DIR)

for split in ["train", "val", "test"]:
    for class_name in class_names:
        os.makedirs(os.path.join(COMBINED_DIR, split, class_name), exist_ok=True)

def copy_file(src, dst):
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy2(src, dst)

for class_name in tqdm(class_names, desc="Splitting real images"):
    real_class_dir = os.path.join(REAL_DIR, class_name)
    image_paths = []
    for ext in ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp"]:
        image_paths.extend(glob.glob(os.path.join(real_class_dir, ext)))
    image_paths = sorted(image_paths)
    random.shuffle(image_paths)
    n_total = len(image_paths)
    n_train = int(n_total * train_ratio)
    n_val = int(n_total * val_ratio)
    train_files = image_paths[:n_train]
    val_files = image_paths[n_train:n_train+n_val]
    test_files = image_paths[n_train+n_val:]
    for src in train_files: copy_file(src, os.path.join(COMBINED_DIR, "train", class_name, "REAL_" + os.path.basename(src)))
    for src in val_files: copy_file(src, os.path.join(COMBINED_DIR, "val", class_name, "REAL_" + os.path.basename(src)))
    for src in test_files: copy_file(src, os.path.join(COMBINED_DIR, "test", class_name, "REAL_" + os.path.basename(src)))
    for src in sorted(glob.glob(os.path.join(SYNTHETIC_DIR, class_name, "*.png"))):
        copy_file(src, os.path.join(COMBINED_DIR, "train", class_name, "SYN_" + os.path.basename(src)))

print("Combined dataset created:", COMBINED_DIR)
print("Train images:", sum(len(files) for _,_,files in os.walk(os.path.join(COMBINED_DIR,"train"))))
print("Val images:", sum(len(files) for _,_,files in os.walk(os.path.join(COMBINED_DIR,"val"))))
print("Test images:", sum(len(files) for _,_,files in os.walk(os.path.join(COMBINED_DIR,"test"))))

In [ ]:
# =========================
# LOAD DATASET FOR VGG19
# =========================
train_transform = transforms.Compose([
    transforms.Resize((cnn_img_size, cnn_img_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])
eval_transform = transforms.Compose([
    transforms.Resize((cnn_img_size, cnn_img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])
train_dataset = datasets.ImageFolder(os.path.join(COMBINED_DIR, "train"), transform=train_transform)
val_dataset = datasets.ImageFolder(os.path.join(COMBINED_DIR, "val"), transform=eval_transform)
test_dataset = datasets.ImageFolder(os.path.join(COMBINED_DIR, "test"), transform=eval_transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
class_names = train_dataset.classes
num_classes = len(class_names)
print("Classes:", num_classes)
print("Train:", len(train_dataset), "Val:", len(val_dataset), "Test:", len(test_dataset))

In [ ]:
# =========================
# BUILD VGG19 TRANSFER LEARNING MODEL
# =========================
try:
    weights = models.VGG19_Weights.IMAGENET1K_V1
    model = models.vgg19(weights=weights)
except Exception:
    model = models.vgg19(pretrained=True)

for idx, param in enumerate(model.features.parameters()):
    param.requires_grad = idx >= frozen_layers

in_features = model.classifier[6].in_features
model.classifier[5] = nn.Dropout(p=dropout_rate)
model.classifier[6] = nn.Linear(in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=learning_rate, weight_decay=weight_decay)
print("VGG19 ready.")

In [ ]:
# =========================
# TRAINING HELPERS
# =========================
def train_one_epoch(model, loader):
    model.train(); running_loss=0.0; correct=0; total=0
    for images, labels in tqdm(loader, desc="Training", leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward(); optimizer.step()
        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item(); total += labels.size(0)
    return running_loss/total, correct/total

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); running_loss=0.0; correct=0; total=0
    all_labels=[]; all_preds=[]; all_probs=[]
    for images, labels in tqdm(loader, desc="Evaluating", leave=False):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        probs = torch.softmax(outputs, dim=1)
        preds = outputs.argmax(1)
        running_loss += loss.item() * images.size(0)
        correct += (preds == labels).sum().item(); total += labels.size(0)
        all_labels.extend(labels.cpu().numpy()); all_preds.extend(preds.cpu().numpy()); all_probs.extend(probs.cpu().numpy())
    return {"loss": running_loss/total, "accuracy": correct/total, "labels": np.array(all_labels), "preds": np.array(all_preds), "probs": np.array(all_probs)}

In [ ]:
# =========================
# TRAIN VGG19
# =========================
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0
best_model_path = os.path.join(MODEL_DIR, "best_vgg19_real_cwgan.pth")

for epoch in range(1, epochs + 1):
    print(f"
Epoch {epoch}/{epochs}")
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_results = evaluate(model, val_loader)
    val_loss, val_acc = val_results["loss"], val_results["accuracy"]
    history["train_loss"].append(train_loss); history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss); history["val_acc"].append(val_acc)
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({"epoch": epoch, "model_state_dict": model.state_dict(), "optimizer_state_dict": optimizer.state_dict(), "best_val_acc": best_val_acc, "class_names": class_names}, best_model_path)
        print("Saved new best model:", best_model_path)
print("Best validation accuracy:", best_val_acc)

In [ ]:
# =========================
# PLOTS
# =========================
epochs_range = range(1, len(history["train_acc"]) + 1)
plt.figure(figsize=(9,6))
plt.plot(epochs_range, history["train_acc"], label="Train Accuracy")
plt.plot(epochs_range, history["val_acc"], label="Validation Accuracy")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.title("VGG19 Accuracy Curve - Real + cWGAN")
plt.yticks(np.arange(0.80, 1.01, 0.05)); plt.ylim(0.80, 1.00)
plt.grid(True); plt.legend()
acc_plot_path = os.path.join(PLOT_DIR, "vgg19_accuracy_curve_real_cwgan.png")
plt.savefig(acc_plot_path, dpi=300, bbox_inches="tight"); plt.show()

plt.figure(figsize=(9,6))
plt.plot(epochs_range, history["train_loss"], label="Train Loss")
plt.plot(epochs_range, history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("VGG19 Loss Curve - Real + cWGAN")
plt.grid(True); plt.legend()
loss_plot_path = os.path.join(PLOT_DIR, "vgg19_loss_curve_real_cwgan.png")
plt.savefig(loss_plot_path, dpi=300, bbox_inches="tight"); plt.show()

In [ ]:
# =========================
# TEST BEST MODEL
# =========================
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
test_results = evaluate(model, test_loader)
y_true, y_pred, y_prob = test_results["labels"], test_results["preds"], test_results["probs"]

test_acc = accuracy_score(y_true, y_pred)
test_precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
test_recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
test_f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print("===== FINAL TEST RESULTS: VGG19 REAL + cWGAN =====")
print(f"Accuracy:  {test_acc:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall:    {test_recall:.4f}")
print(f"F1-score:  {test_f1:.4f}")
print("
Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

In [ ]:
# =========================
# CONFUSION MATRIX
# =========================
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(14,12))
plt.imshow(cm, interpolation="nearest")
plt.title("Confusion Matrix - VGG19 Real + cWGAN")
plt.colorbar()
tick_marks = np.arange(len(class_names))
plt.xticks(tick_marks, class_names, rotation=90, fontsize=7)
plt.yticks(tick_marks, class_names, fontsize=7)
plt.xlabel("Predicted Label"); plt.ylabel("True Label"); plt.tight_layout()
cm_path = os.path.join(PLOT_DIR, "vgg19_confusion_matrix_real_cwgan.png")
plt.savefig(cm_path, dpi=300, bbox_inches="tight"); plt.show()

In [ ]:
# =========================
# ROC-AUC CURVE
# =========================
y_true_bin = label_binarize(y_true, classes=list(range(num_classes)))
try:
    micro_auc = roc_auc_score(y_true_bin, y_prob, average="micro", multi_class="ovr")
    macro_auc = roc_auc_score(y_true_bin, y_prob, average="macro", multi_class="ovr")
    print(f"Micro ROC-AUC: {micro_auc:.4f}")
    print(f"Macro ROC-AUC: {macro_auc:.4f}")
    fpr_micro, tpr_micro, _ = roc_curve(y_true_bin.ravel(), y_prob.ravel())
    roc_auc_micro = auc(fpr_micro, tpr_micro)
    plt.figure(figsize=(8,6))
    plt.plot(fpr_micro, tpr_micro, label=f"Micro-average ROC curve (AUC = {roc_auc_micro:.4f})")
    plt.plot([0,1], [0,1], linestyle="--")
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title("ROC-AUC Curve - VGG19 Real + cWGAN")
    plt.grid(True); plt.legend(loc="lower right")
    roc_path = os.path.join(PLOT_DIR, "vgg19_roc_auc_real_cwgan.png")
    plt.savefig(roc_path, dpi=300, bbox_inches="tight"); plt.show()
except Exception as e:
    print("ROC-AUC could not be computed.")
    print("Reason:", e)

In [ ]:
# =========================
# SAVE SUMMARY RESULTS
# =========================
summary_path = os.path.join(OUTPUT_DIR, "vgg19_real_cwgan_summary.txt")
with open(summary_path, "w") as f:
    f.write("VGG19 Real + cWGAN Results
")
    f.write("==========================

")
    f.write(f"Generator checkpoint: {GENERATOR_CHECKPOINT}
")
    f.write(f"Synthetic images per class: {synthetic_per_class}
")
    f.write(f"Combined dataset: {COMBINED_DIR}

")
    f.write("Hyperparameters
")
    f.write("----------------
")
    f.write(f"Optimizer: {optimizer_name}
")
    f.write(f"Epochs: {epochs}
")
    f.write(f"Initial learning rate: {learning_rate}
")
    f.write(f"L2 regularizer: {weight_decay}
")
    f.write(f"Batch size: {batch_size}
")
    f.write(f"Dropout rate: {dropout_rate}
")
    f.write(f"Frozen layers: {frozen_layers}

")
    f.write("Final Test Results
")
    f.write("------------------
")
    f.write(f"Accuracy: {test_acc:.4f}
")
    f.write(f"Precision: {test_precision:.4f}
")
    f.write(f"Recall: {test_recall:.4f}
")
    f.write(f"F1-score: {test_f1:.4f}
")
    if 'micro_auc' in globals():
        f.write(f"Micro ROC-AUC: {micro_auc:.4f}
")
        f.write(f"Macro ROC-AUC: {macro_auc:.4f}
")
print("Saved summary:", summary_path)
print("All outputs saved in:", OUTPUT_DIR)